# 🎭 Day 2 — PCA in Action: Compress · Reconstruct · Visualize
### Unsupervised Machine Learning | In-Class Activity

---

**Day 1 recap:** Eigenvectors of the covariance matrix point in directions of maximum variance. Eigenvalues measure how much variance each direction captures.

> **Today's Big Question:** *If we describe a face using only 50 "eigenface coordinates" instead of 4096 pixels — how good is the reconstruction, and what else can we do in that compressed space?*

---

### How each step works
| Block | What you do |
|-------|-------------|
| 🟢 **PROVIDED** | Read & run the cell — no changes needed |
| ✏️ **Activity 1 — CODE** | Fill every `___` blank (~75% of the code) |
| 🧠 **Activity 2 — CONCEPT** | Answer in the Markdown cell below the question |
| 🔍 **Activity 3 — INTERPRET** | Run your code, then write what you observe |

---

## 📦 Step 0 — Setup & Reload
🟢 **PROVIDED** — run this cell to reload the dataset and recompute SVD from Day 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

faces      = fetch_olivetti_faces(shuffle=True, random_state=42)
X, images, labels = faces.data, faces.images, faces.target

mean_face  = X.mean(axis=0)
X_centered = X - mean_face

U, S, Vt        = np.linalg.svd(X_centered, full_matrices=False)
components       = Vt                              # rows = eigenvectors
eigenvalues_full = (S**2) / (X_centered.shape[0] - 1)

print("✅ Ready! X shape:", X.shape)

---
## 📉 Step 1 — Project Faces into PCA Space

**Theory:** Projection compresses each face from 4096 numbers to k numbers — its coordinates in PCA space.

```
scores = X_centered · Vₖᵀ
```

where Vₖ = the top k rows of Vt (our eigenvectors).

🟢 **PROVIDED** — note below.

In [ ]:
# We use k = 50 components in this step

### ✏️ Activity 1 — CODE
**Task:** Select the top 50 eigenvectors from `components` (Vt). Project all 400 faces. Print original dimension, compressed dimension, compression ratio, and first 5 scores for face #0.

In [ ]:
# --- YOUR CODE (~8 lines) ---

k   = 50
V_k = components[:___, :]         # top k rows → shape (k, 4096)

# scores = X_centered @ V_k.T
scores = X_centered @ ___          # shape (400, k)

print(f"Original dimension   : {X.shape[___]} pixels")
print(f"Compressed dimension : {scores.shape[___]} PCA scores")
print(f"Compression ratio    : {X.shape[1] / k:.0f}× smaller")
print(f"\nFace #0, first 5 scores: {np.round(scores[0, :5], 3)}")

# ── HINT ────────────────────────────────────────────────────────────────
# V_k    = components[:k, :]          # select first k ___
# scores = X_centered @ ___           # transpose of what?
# print(f"Original: {X.shape[___]}")  # index for pixel dimension
# print(f"Compressed: {scores.shape[___]}")  # index for score dimension
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** After projection, each face is described by 50 numbers instead of 4096. What do these 50 numbers actually mean — what is each number measuring physically?

**Your answer:**

> Each score measures how much that face aligns with…

### 🔍 Activity 3 — INTERPRET
Run: `print(scores[0, :10])` and `print(scores[1, :10])`.

**Question:** Are the score vectors for two different people very different or similar? What does a large difference in scores tell you about those two faces?

**Your answer:**

> scores[0] and scores[1] look…  
> A large difference means those faces…

---
## 🔄 Step 2 — Reconstruct Faces from PCA Scores

**Theory:** Reconstruction reverses the projection — going from k scores back to 4096 pixels:

```
X̂ = scores · Vₖ + mean_face
```

More components k = closer to original.

🟢 **PROVIDED** — note below.

In [ ]:
# V_k and scores are available from Step 1 (k=50)

### ✏️ Activity 1 — CODE
**Task:** Complete the `reconstruct()` function. Then plot a **2×5 grid**: original face top-left, reconstructions with k = [1, 5, 10, 20, 50, 100, 200, 400]. Show MSE in each title.

In [ ]:
# --- YOUR CODE (~20 lines) ---

def reconstruct(face_idx, k):
    Vk    = components[:k]
    score = X_centered[face_idx] @ Vk.___     # project 1 face: (4096,) → (k,)
    recon = score @ ___ + ___                  # reconstruct: (k,) → (4096,)
    return recon.reshape(64, 64)

face_idx = 7   # change this to explore a different face
k_list   = [1, 5, 10, 20, 50, 100, 200, 400]

fig, axes = plt.subplots(2, 5, figsize=(17, 7))
fig.suptitle("Face Reconstruction vs Number of Components", fontsize=13)

axes[0, 0].imshow(images[___], cmap="gray")
axes[0, 0].set_title("ORIGINAL\n4096 px", color="green", fontweight="bold")
axes[0, 0].axis("off")

for idx, k in enumerate(k_list):
    row, col = divmod(idx + 1, 5)
    recon = reconstruct(___, ___)
    mse   = np.mean((images[face_idx] - ___) ** 2)
    axes[row, col].imshow(np.clip(recon, 0, 1), cmap="gray")
    axes[row, col].set_title(f"k={k}\nMSE={mse:.5f}", fontsize=9)
    axes[row, col].axis("off")

plt.tight_layout(); plt.show()

# ── HINT ────────────────────────────────────────────────────────────────
# score = X_centered[face_idx] @ Vk.___  # .T or nothing?
# recon = score @ ___ + ___              # Vk or Vk.T? then + what?
# axes[0,0].imshow(images[___], cmap="gray")   # which index?
# mse = np.mean((images[face_idx] - ___)**2)   # recon goes where?
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** With k=1 the face is a blur. With k=400 it is perfect. Explain **mathematically** why k=400 gives exact reconstruction while k=1 is very lossy.

**Your answer:**

> With k=400 we use all ___ components, which means…  
> With k=1 we lose ___% of variance because…

### 🔍 Activity 3 — INTERPRET
Look at your reconstruction grid carefully.

**Question:** At what k does the face become **recognizable as a specific person**? At what k does it look **"good enough"**? Are those two k values the same?

**Your answer:**

> Recognizable at k ≈ ___  
> Good enough at k ≈ ___  
> Same? Yes/No because…

---
## 📈 Step 3 — Reconstruction Error Curve

**Theory:** Plotting MSE vs k helps us choose the right number of components systematically.  
We look for the **"elbow"** — where adding more components gives diminishing error reduction.

🟢 **PROVIDED** — run this.

In [ ]:
k_range = list(range(1, 51)) + list(range(55, 401, 10))
errors  = []

### ✏️ Activity 1 — CODE
**Task:** For each k in `k_range`, reconstruct **ALL 400 faces** and compute average MSE across all faces. Append to `errors`. Plot MSE vs k with vertical lines at k=20, 50, 100.

In [ ]:
# --- YOUR CODE (~16 lines) ---

for k in k_range:
    Vk      = components[:___, :]
    sc      = X_centered @ Vk.___           # project all 400 faces
    X_recon = sc @ ___ + ___               # reconstruct all 400 faces
    mse     = np.mean((___ - X_recon) ** 2)
    errors.append(___)

plt.figure(figsize=(9, 4))
plt.plot(___, ___, color="steelblue", lw=2)

for kv, col, lbl in [(20,"green","k=20"),(50,"red","k=50"),(100,"orange","k=100")]:
    plt.axvline(___, color=col, ls="--", alpha=0.8, label=___)

plt.xlabel("Number of Components k")
plt.ylabel("Avg MSE (all 400 faces)")
plt.title("📉 Reconstruction Error vs k")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── HINT ────────────────────────────────────────────────────────────────
#     Vk      = components[:k, :]
#     sc      = X_centered @ ___      # Vk.T
#     X_recon = sc @ Vk + ___         # add mean_face
#     mse     = np.mean((X - ___)**2) # X_recon
# plt.plot(k_range, ___, ...)         # what list on y-axis?
# plt.axvline(kv, ..., label=___)     # lbl goes here
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** The MSE curve flattens but **never reaches exactly 0** even at k=400. Why? Think about the maximum possible rank of `X_centered` given its dimensions (400 rows, 4096 cols).

**Your answer:**

> MSE never reaches 0 because the rank of X_centered is at most ___, which means…

### 🔍 Activity 3 — INTERPRET
Examine your error plot.

**Question:** Where is the **elbow**? Based on this curve — what k would you recommend? Does this agree with your Day 1 cumulative variance answer?

**Your answer:**

> Elbow is around k = ___  
> I recommend k = ___  
> Agrees / disagrees with Day 1 because…

---
## ⚡ Step 4 — sklearn PCA: The Clean API

**Theory:** `sklearn`'s `PCA` wraps everything into `fit_transform()` and `inverse_transform()`.  
It is faster, handles edge cases, and is what you'll use in real projects.  
Let's verify it matches our manual results.

🟢 **PROVIDED** — run this.

In [ ]:
from sklearn.decomposition import PCA

### ✏️ Activity 1 — CODE
**Task:** Create PCA with 50 components. Fit and transform X. Print top-component variance %, total variance %, and compare sklearn's top 3 eigenvalues to our manual ones. Reconstruct 6 faces and show original vs reconstructed.

In [ ]:
# --- YOUR CODE (~20 lines) ---

pca   = PCA(n_components=___, random_state=42)
X_pca = pca.___(X)          # fit AND transform in one call

print(f"Top component variance : {pca.explained_variance_ratio_[___]*100:.2f}%")
print(f"Total variance (k=50)  : {pca.explained_variance_ratio_.___()*100:.2f}%")

print("\n--- Eigenvalue comparison ---")
print(f"Manual top 3  : {np.round(eigenvalues_full[:3], 4)}")
print(f"sklearn top 3 : {np.round(pca.explained_variance_[:3], 4)}")

# Reconstruct
X_recon_sk = pca.___(X_pca)   # inverse_transform

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle("sklearn PCA Reconstruction (k=50)")
for i in range(6):
    axes[0, i].imshow(X[i].reshape(64, 64), cmap="gray")
    axes[0, i].set_title("Original", fontsize=8); axes[0, i].axis("off")
    axes[1, i].imshow(___[i].reshape(64, 64), cmap="gray")
    axes[1, i].set_title("k=50", fontsize=8); axes[1, i].axis("off")
plt.tight_layout(); plt.show()

# ── HINT ────────────────────────────────────────────────────────────────
# pca   = PCA(n_components=50, ...)
# X_pca = pca.fit_transform(___)          # what data goes in?
# pca.explained_variance_ratio_[0]        # index for top component
# pca.explained_variance_ratio_.sum()     # method to total them
# X_recon_sk = pca.inverse_transform(___) # what do we pass?
# axes[1,i].imshow(___[i].reshape(64,64), ...)  # which array?
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** sklearn uses a **randomized SVD** by default, not the exact SVD we ran. Name one **advantage** and one **disadvantage** of the randomized approach.

**Your answer:**

> Advantage: …  
> Disadvantage: …

### 🔍 Activity 3 — INTERPRET
Compare the manual and sklearn eigenvalues you printed.

**Question:** Are they identical or slightly different? If different — why might that happen even though both methods solve the same theoretical problem?

**Your answer:**

> They are ___ because…

---
## 🗺️ Step 5 — Visualize Face Space in 2D

**Theory:** Projecting 400 faces onto just 2 components lets us plot them in 2D.  
If PCA captures meaningful structure, photos of the **same person should cluster** together.

🟢 **PROVIDED** — run this.

In [ ]:
pca_2d = PCA(n_components=2)
X_2d   = pca_2d.fit_transform(X)    # all 400 faces → 2D coordinates

### ✏️ Activity 1 — CODE
**Task:** Create a scatter plot of `X_2d` colored by person label. Add a colorbar. Label axes with PC number and variance %. Highlight one specific person (e.g. person 3) with larger black-outlined markers.

In [ ]:
# --- YOUR CODE (~18 lines) ---

fig, ax = plt.subplots(figsize=(11, 8))

# All 400 faces
sc = ax.scatter(
    X_2d[:, ___], X_2d[:, ___],
    c=___, cmap="tab20",
    s=40, alpha=0.6, edgecolors="white", linewidths=0.3
)
plt.colorbar(sc, ax=ax, label="Person ID", shrink=0.8)

# Highlight one person
highlight_id = 3
mask = labels == ___
ax.scatter(
    X_2d[mask, 0], X_2d[mask, ___],
    s=150, edgecolors="black", linewidths=1.5,
    facecolors="none", label=f"Person {highlight_id}")

ax.set_xlabel(f"PC1 — {pca_2d.explained_variance_ratio_[___]*100:.1f}% variance")
ax.set_ylabel(f"PC2 — {pca_2d.explained_variance_ratio_[___]*100:.1f}% variance")
ax.set_title("400 Faces in 2D PCA Space  (color = person ID)")
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

# ── HINT ────────────────────────────────────────────────────────────────
# sc = ax.scatter(X_2d[:, 0], X_2d[:, ___], c=___, ...)  # PC2 col, labels
# mask = labels == highlight_id
# ax.scatter(X_2d[mask, 0], X_2d[mask, ___], ...)         # PC2 col
# ax.set_xlabel(f"PC1 — {pca_2d.explained_variance_ratio_[0]*100:.1f}%...")
# ax.set_ylabel(f"PC2 — {pca_2d.explained_variance_ratio_[___]*100:.1f}%...")
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** PC1 + PC2 explain only a small % of total variance (check your axis labels). Does that make this 2D plot **misleading**? When is 2D PCA visualization useful and when is it not?

**Your answer:**

> The 2D plot shows ___% of variance.  
> Useful when…  
> Not useful / misleading when…

### 🔍 Activity 3 — INTERPRET
Look at the scatter plot and your highlighted person.

**Question:** Are that person's 10 photos clustered tightly or spread out? Pick two people whose dots are close together in PCA space — do you predict their faces look similar? Why?

**Your answer:**

> Person ___ photos are clustered / spread because…  
> Persons ___ and ___ are close in PCA space, predicting…

---
## 👯 Step 6 — Nearest Neighbor Face Search

**Theory:** In PCA space, **Euclidean distance ≈ visual similarity**.  
The nearest neighbor to a query face (smallest L2 distance) should be the most visually similar face in the dataset.

🟢 **PROVIDED** — run this.

In [ ]:
pca50 = PCA(n_components=50)
X_50  = pca50.fit_transform(X)   # all 400 faces in 50-D PCA space

### ✏️ Activity 1 — CODE
**Task:** Set face #0 as query. Compute L2 distance to every other face in PCA space. Set `distance[0]=∞` to exclude itself. Find top 4 nearest neighbors. Display query + neighbors, marking whether each is the same person (✅/❌).

In [ ]:
# --- YOUR CODE (~18 lines) ---

query_idx = 0
query_vec = X_50[___]

dists = np.linalg.norm(X_50 - ___, axis=___)   # L2 from query to all faces
dists[___] = np.inf                             # exclude the query itself

top4 = np.argsort(___)[___]                     # sorted indices, first 4

fig, axes = plt.subplots(1, 5, figsize=(13, 3))

axes[0].imshow(images[___], cmap="gray")
axes[0].set_title(f"QUERY\nPerson {labels[query_idx]}", color="blue", fontweight="bold")
axes[0].axis("off")

for j, idx in enumerate(___):
    axes[j+1].imshow(images[___], cmap="gray")
    match = "✅" if labels[idx] == labels[___] else "❌"
    axes[j+1].set_title(
        f"#{j+1} {match}\nPerson {labels[idx]}\nd={dists[idx]:.2f}", fontsize=9)
    axes[j+1].axis("off")

plt.suptitle("PCA Nearest Neighbor Search", fontsize=12)
plt.tight_layout(); plt.show()

# ── HINT ────────────────────────────────────────────────────────────────
# query_vec = X_50[0]
# dists = np.linalg.norm(X_50 - ___, axis=1)   # broadcast query_vec
# dists[0]  = np.inf
# top4 = np.argsort(dists)[:___]               # take first how many?
# for j, idx in enumerate(top4):
#     axes[j+1].imshow(images[___], ...)        # images[idx]
#     match = "✅" if labels[idx] == labels[___] else "❌"  # query label
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** If we used pixel-space L2 distance (no PCA) instead, would results likely be **better or worse**? What is the "curse of dimensionality" and how does PCA help with nearest neighbor search?

**Your answer:**

> Pixel-space distance would be ___ because…  
> Curse of dimensionality means…  
> PCA helps by…

### 🔍 Activity 3 — INTERPRET
Run the search and look at your results. Try a few different values of `query_idx`.

**Question:** How many of the top 4 neighbors are the same person as the query (count ✅)? Did you find any failure cases? Describe one.

**Your answer:**

> For query #0: ___ / 4 matches  
> For query #___: ___ / 4 matches  
> A failure/success case: …

---
## ✨ Step 7 — Generate a Synthetic Face

**Theory:** If we average the PCA coordinates of all photos of one person, then reconstruct with `inverse_transform()`, we create a **synthetic face** — one that was never photographed.  
It's the "PCA average" of that person.

🟢 **PROVIDED** — note below.

In [ ]:
# pca50 and X_50 are available from Step 6

### ✏️ Activity 1 — CODE
**Task:** Write a `make_synthetic(person_id)` function that finds all photos of that person, averages their 50-D PCA coordinates, and reconstructs. Call it for two different people. Display real photos + synthetic for each, then compare both synthetics side by side.

In [ ]:
# --- YOUR CODE (~25 lines) ---

def make_synthetic(person_id):
    idx     = np.where(labels == ___)[0]
    avg_pca = X_50[___].mean(axis=___)              # average over photos
    synth   = pca50.inverse_transform(___.reshape(1, -1))
    return idx, synth.reshape(64, 64)

person_a, person_b = 3, 7
idx_a, synth_a = make_synthetic(___)
idx_b, synth_b = make_synthetic(___)

for pid, idx, synth in [(person_a, idx_a, synth_a), (person_b, idx_b, synth_b)]:
    fig, axes = plt.subplots(1, len(idx) + 1, figsize=(14, 3))
    fig.suptitle(f"Person {pid}", fontsize=11)
    for j, i in enumerate(___):
        axes[j].imshow(images[___], cmap="gray")
        axes[j].set_title(f"Real #{j+1}", fontsize=8); axes[j].axis("off")
    axes[-1].imshow(np.clip(___, 0, 1), cmap="gray")
    axes[-1].set_title("✨ Synthetic", color="purple", fontweight="bold", fontsize=8)
    axes[-1].axis("off")
    plt.tight_layout(); plt.show()

# Side-by-side synthetic comparison
fig, axes = plt.subplots(1, 2, figsize=(5, 3))
axes[0].imshow(np.clip(synth_a, 0, 1), cmap="gray")
axes[0].set_title(f"Synth A — Person {person_a}"); axes[0].axis("off")
axes[1].imshow(np.clip(___, 0, 1), cmap="gray")
axes[1].set_title(f"Synth B — Person {person_b}"); axes[1].axis("off")
plt.suptitle("Synthetic Face Comparison"); plt.tight_layout(); plt.show()

# ── HINT ────────────────────────────────────────────────────────────────
# def make_synthetic(person_id):
#     idx     = np.where(labels == person_id)[0]
#     avg_pca = X_50[idx].mean(axis=___)          # axis=0 collapses photos
#     synth   = pca50.inverse_transform(___.reshape(1, -1))  # avg_pca
#     return idx, synth.reshape(64, ___)          # second dim?
# for j, i in enumerate(idx):
#     axes[j].imshow(images[___], ...)            # i goes where?
# axes[-1].imshow(np.clip(synth, ___, ___), ...)  # clip range [0, 1]
# ────────────────────────────────────────────────────────────────────────

### 🧠 Activity 2 — CONCEPT
**Question:** The synthetic face is generated by linear interpolation in PCA space. What is the **fundamental limitation** of this approach compared to modern generative models (GANs, diffusion models)?

**Your answer:**

> PCA synthesis works by…  
> The limitation is that PCA can only produce…  
> GANs/diffusion models can do ___ which PCA cannot…

### 🔍 Activity 3 — INTERPRET
Compare the two synthetic faces side by side.

**Question:** Do the two synthetic faces look clearly different from each other? List one feature the synthetic **gets right** and one it **fails to capture**.

**Your answer:**

> Synthetic A and B look ___ from each other.  
> Gets right: …  
> Fails to capture: …

---

## 🎉 PCA Unit Complete!

You built PCA from scratch and applied it end-to-end:

| Step | What you did |
|------|-------------|
| Day 1 — 1 | Loaded face data, explored pixel representation |
| Day 1 — 2 | Computed mean face and centered data |
| Day 1 — 3 | Built the covariance matrix manually |
| Day 1 — 4 | Computed eigenvalues and eigenvectors |
| Day 1 — 5 | Visualized eigenvectors as arrows on data |
| Day 1 — 6 | Revealed Eigenfaces from full 4096-pixel data |
| Day 1 — 7 | Analyzed the eigenvalue spectrum |
| Day 2 — 1 | Projected faces into compressed PCA space |
| Day 2 — 2 | Reconstructed faces at varying k |
| Day 2 — 3 | Plotted reconstruction error curve |
| Day 2 — 4 | Used sklearn PCA and verified against manual |
| Day 2 — 5 | Visualized 400 faces in 2D PCA space |
| Day 2 — 6 | Found most similar faces via nearest neighbor |
| Day 2 — 7 | Generated synthetic faces via PCA averaging |

*Dataset: Olivetti Faces — AT&T Laboratories Cambridge*